# RDF synthetic ticket corpus — executed integrity checks

**Scope:** 12 intentionally synthetic cases. Does not perform a live ElevenLabs or Raindrop call, nor does it benchmark agent quality. Atlas upload/readback is a separate connector operation.

In [1]:
from pathlib import Path
import csv,json,hashlib,collections,platform,sys,re
p=Path.cwd(); print("Python",platform.python_version(),"CWD",p.name)

Python 3.13.5 CWD rdf_hackathon_handoff_v1


In [2]:
with (p/"raindrop_eval_dataset.csv").open(encoding="utf-8",newline="") as f:
    rows=list(csv.DictReader(f))
with (p/"atlas_ticket_fixtures.jsonl").open(encoding="utf-8") as f:
    atlas=[json.loads(s) for s in f if s.strip()]
assert len(rows)==len(atlas)==12
assert len({r["case_id"] for r in rows})==12
print("PASS: 12 rows and unique IDs")

PASS: 12 rows and unique IDs


In [3]:
counts=dict(sorted(collections.Counter(r["split"] for r in rows).items()))
assert counts=={"adversarial":2,"dev":6,"heldout":4}
assert all(r["input"] and r["source_version"]=="RDF-SYN-TICKET-20261011-V1" for r in rows)
assert [d["_id"] for d in atlas]==[r["case_id"] for r in rows]
assert all(d["synthetic"] and d["project_id"]=="RDF-SFTW26-NEW-001" for d in atlas)
print("PASS: split, source version, Atlas mapping",counts)

PASS: split, source version, Atlas mapping {'adversarial': 2, 'dev': 6, 'heldout': 4}


In [4]:
assert all(r["input"].strip() for r in rows)
assert all(d["status"]=="UNEXECUTED_SCENARIO" for d in atlas)
assert not any("@" in r["input"] or re.search(r"(?:\+\d{1,3}[- .]?)?\(?\d{3}\)?[- .]?\d{3}[- .]?\d{4}",r["input"]) for r in rows)
assert next(r for r in rows if r["case_id"]=="RDF-SYN-011")["expected_action"]=="DENY_AND_ESCALATE"
assert next(r for r in rows if r["case_id"]=="RDF-SYN-008")["expected_action"]=="TEXT_ROUTE"
print("PASS: nonempty current messages, basic PII red flags and safety labels (not a full DLP audit)")

PASS: nonempty current messages, basic PII red flags and safety labels (not a full DLP audit)


In [5]:
raw=(p/"atlas_ticket_fixtures.jsonl").read_bytes()
digest=hashlib.sha256(raw).hexdigest()
reconstructed="".join(json.dumps(d,sort_keys=True,separators=(",",":"),ensure_ascii=False)+"\n" for d in atlas).encode("utf-8")
assert raw==reconstructed
print("PASS: deterministic JSONL serialization; SHA256",digest)

PASS: deterministic JSONL serialization; SHA256 e03beb1a5118bee5f8532939043b7fa69092cc527cf600cba68f109c4f952dbe


In [6]:
events=[{"_id":"delivery-1","conversation_id":"mock-1","type":"post_call_transcription"},{"_id":"delivery-1-duplicate","conversation_id":"mock-1","type":"post_call_transcription"}]
keys={e["type"]+":"+e["conversation_id"] for e in events}
assert len(keys)==1
print("PASS: synthetic duplicate event collapse via type + conversation_id; not a live webhook test")

PASS: synthetic duplicate event collapse via type + conversation_id; not a live webhook test


In [7]:
print("CHECK_SUMMARY: PASS=5; LIVE_ELEVENLABS=NOT_TESTED; RAINDROP_CLOUD=NOT_TESTED; ATLAS_WRITE=SEPARATE_RECEIPT; MODEL_IMPROVEMENT=NOT_TESTED")

CHECK_SUMMARY: PASS=5; LIVE_ELEVENLABS=NOT_TESTED; RAINDROP_CLOUD=NOT_TESTED; ATLAS_WRITE=SEPARATE_RECEIPT; MODEL_IMPROVEMENT=NOT_TESTED
